# 집계 자료만으로 시작하는 IPU

표본 미시 자료가 없고 공표된 집계표만 있는 상황에서 가중치 조정을 시작하는 절차를 다룬다. 조정할 개체가 주어지지 않으므로, 개체 자체를 **열거**하여 만들어 낸다. 가능한 속성 조합을 모두 나열하여 각 조합을 기본 단위로 삼고, 그 조합에 해당하는 개체가 모집단에 몇 개 있는지를 가중치로 추정한다.

표본에서 출발하는 절차는 [`01_기본_사용법.ipynb`](01_기본_사용법.ipynb)에서 다룬다. 두 노트북은 같은 엔진을 쓰지만 출발점이 다르며, 그 차이가 만드는 결과를 이 노트북에서 확인한다.

| 이 노트북에서 다루는 것 | 절 |
| --- | --- |
| 공표된 집계표들이 서로 양립하는지 미리 검사한다 | 3 |
| 미공표 셀의 목표를 균형식으로 연역한다 | 4 |
| 가능한 속성 조합을 열거하여 기본 단위를 만든다 | 5 |
| 구조적 영이 제약 열이 아니라 기본 단위를 제거한다 | 5 |
| 독립 가정으로 초기 가중치를 놓는다 | 9 |
| 열거 표본에는 표본 영이 발생하지 않는다 | 9 |
| 개방 계급의 내부 구성이 다른 제약으로 결정된다 | 11 |
| 제약이 결정하지 못하는 결합 구조가 남는다 | 12, 13 |
| 정수화로 합성 모집단을 만든다 | 14 |

### 검증 방침

각 절에서 확인한 사실을 `check()` 로 기록한다. 조건이 거짓이면 그 자리에서 `AssertionError` 가 발생하므로, 노트북이 끝까지 실행되었다는 것 자체가 모든 검증을 통과했다는 뜻이다. 마지막 절에서 기록을 표로 모아 확인한다.

용어와 기호는 [`docs/01_용어와_정의.md`](../docs/01_용어와_정의.md)를 따른다.

## 0. 준비

라이브러리와 표준 도구를 불러온다. 저장소 최상위에서 `pip install -e .` 를 먼저 실행해야 한다.

In [1]:
from itertools import combinations_with_replacement
from math import comb, factorial

import numpy as np
import pandas as pd

import generalized_ipu as gipu

print("generalized-ipu", gipu.__version__)

CHECKS = []


def check(requirement, description, condition):
    # 검증 결과를 기록한다. 조건이 거짓이면 즉시 예외를 발생시킨다.
    passed = bool(condition)
    CHECKS.append({"문제": requirement, "검증 내용": description, "통과": passed})
    if not passed:
        raise AssertionError(f"검증 실패: {description}")
    return passed

generalized-ipu 0.1.0.dev0


## 1. 표본이 없을 때 무엇이 달라지는가

반복 갱신식은 그대로이다. 달라지는 것은 속성 행렬 $A$ 의 행이 어디에서 오는가, 그리고 가중치 $w_i$ 를 무엇으로 읽는가이다.

| 항목 | 표본에서 시작 | 집계 자료만으로 시작 |
| --- | --- | --- |
| 기본 단위 | 관측된 개체 하나 | 가능한 속성 조합 하나, 곧 **유형** |
| 기본 단위 수 $n$ | 표본 크기 | 열거된 유형의 수 |
| 가중치 $w_i$ | 그 개체가 대표하는 모집단 개체 수 | 그 유형에 해당하는 모집단 개체 수 |
| 초기 가중치 | 추출률의 역수 | 주변표의 곱, 곧 독립 가정 |
| 결합 구조의 출처 | 표본에서 관측된 공기 빈도 | 제약과 초기값이 함께 결정 |
| 구조적 영 | 제약 열을 제외 | **기본 단위를 열거하지 않음** |
| 표본 영 | 발생하며 평활이 필요 | 발생하지 않음 |

마지막 세 줄이 이 노트북의 핵심이다. 표본이 없으면 관측된 공기 빈도가 없으므로, 제약이 지정하지 않은 결합 구조는 자료가 아니라 초기값이 결정한다. 이 사실은 12절에서 두 초기값의 해를 나란히 놓아 확인한다.

**유효 마스크의 축에 주의한다.** `LogicalRuleParser` 가 만드는 유효 마스크는 길이가 제약 열의 수 $K$ 이며 제약 열을 걸러 낸다. 이 노트북의 도메인 규칙은 열이 아니라 기본 단위를 걸러 내므로 축이 다르다. 따라서 마스크를 쓰지 않고 열거 단계에서 해당 조합을 만들지 않는 방식으로 처리한다.

## 2. 공표 집계표

가지고 있는 자료는 다음 네 개의 표뿐이다. 미시 자료는 어디에도 없다.

| 표 | 출처로 가정한 통계 | 성격 |
| --- | --- | --- |
| 지역 × 가구원 수 계급별 가구 수 | 인구주택총조사 | 고정 스칼라 |
| 전국 연령 × 성별 인구 | 인구주택총조사 | 고정 스칼라 |
| 지역 × 연령 대범주 인구 | 인구주택총조사 | 고정 스칼라 |
| 차량 대수별 가구 수 | 가구 조사, 1대와 2대가 미공표 | 4절에서 구간으로 연역 |
| 총 차량 대수 | 자동차 등록 통계, 기준 시점 차이를 구간으로 | 구간 |

가구원 수 계급의 `4+` 는 상한이 없는 **개방 계급**이다. 공표표는 이 계급의 내부 구성을 알려주지 않으며, 열거에서는 최대 가구원 수를 6명으로 둔다. 이 계급에 4인, 5인, 6인 가구가 어떤 비율로 들어가는지는 11절에서 다른 제약이 결정한다.

In [2]:
REGIONS = ["r1", "r2"]
AGE_GROUPS = ["child", "adult", "senior"]
SEXES = ["M", "F"]
SIZE_CLASSES = ["1", "2", "3", "4+"]
CAR_CLASSES = ["0", "1", "2"]
MAX_SIZE = 6  # 열거에서 허용하는 최대 가구원 수

# 지역 × 가구원 수 계급별 가구 수
hh_region_size = pd.DataFrame(
    [[120000.0, 100000.0, 60000.0, 40000.0],
     [80000.0, 70000.0, 30000.0, 20000.0]],
    index=REGIONS,
    columns=SIZE_CLASSES,
)

# 전국 연령 × 성별 인구
person_age_sex = pd.DataFrame(
    [[108000.0, 102000.0],
     [335000.0, 345000.0],
     [78000.0, 100000.0]],
    index=AGE_GROUPS,
    columns=SEXES,
)

# 지역 × 연령 대범주 인구
person_region_age = pd.DataFrame(
    [[130000.0, 435000.0, 105000.0],
     [80000.0, 245000.0, 73000.0]],
    index=REGIONS,
    columns=AGE_GROUPS,
)

# 차량 대수별 가구 수. 1대와 2대는 공표되지 않았다.
car_margin = pd.Series({"0": 130000.0, "1": np.nan, "2": np.nan})

# 자동차 등록 통계의 총 차량 대수. 가구 조사와 기준 시점이 달라 구간으로 받는다.
CAR_TOTAL = (505000.0, 515000.0)

TOTAL_HOUSEHOLDS = float(hh_region_size.to_numpy().sum())
TOTAL_PERSONS = float(person_age_sex.to_numpy().sum())

print(f"총 가구 {TOTAL_HOUSEHOLDS:,.0f}호, 총 인구 {TOTAL_PERSONS:,.0f}명")
hh_region_size

총 가구 520,000호, 총 인구 1,068,000명


,1,2,3,4+
r1,120000.0,100000.0,60000.0,40000.0
r2,80000.0,70000.0,30000.0,20000.0


## 3. 제약표 사이의 정합성 사전 검사

표본에서 시작할 때는 목표가 하나의 모집단에서 나오므로 표들이 저절로 맞물린다. 집계 자료만으로 시작하면 표마다 출처와 기준 시점이 다르므로, 표들이 서로 양립하는지를 먼저 확인해야 한다. 어긋난 표를 그대로 넣으면 반복은 최대 반복에 도달하고, 그 원인이 알고리즘에 있는지 자료에 있는지 구별하기 어려워진다.

확인할 조건은 두 종류이다.

1. **총계의 일치.** 지역별 인구의 합은 전국 인구와 같아야 하고, 지역별 연령 인구의 합은 전국 연령 인구와 같아야 한다.
2. **열거 공간과의 양립.** 가구 규모 표가 정한 가구 수로 만들 수 있는 인구에는 하한과 상한이 있다. 계급 `4+` 를 모두 4인으로 채우면 최소, 모두 6인으로 채우면 최대이다. 공표된 총 인구가 이 범위 밖이면 어떤 열거로도 두 표를 동시에 만족시킬 수 없다.

In [3]:
check(
    "제약표 정합성",
    "지역별 인구의 합계가 전국 연령×성별 표의 총계와 일치한다",
    np.isclose(person_region_age.to_numpy().sum(), TOTAL_PERSONS),
)
check(
    "제약표 정합성",
    "지역별 연령 대범주의 합계가 전국 연령별 인구와 일치한다",
    np.allclose(
        person_region_age.sum(axis=0).to_numpy(), person_age_sex.sum(axis=1).to_numpy()
    ),
)
check(
    "제약표 정합성",
    "지역별 가구 수의 합계가 총 가구 수와 일치한다",
    np.isclose(hh_region_size.sum(axis=1).sum(), TOTAL_HOUSEHOLDS),
)

# 개방 계급을 최소 4인, 최대 6인으로 채웠을 때의 인구 범위
lower_sizes = np.array([1.0, 2.0, 3.0, 4.0])
upper_sizes = np.array([1.0, 2.0, 3.0, float(MAX_SIZE)])

feasibility = pd.DataFrame(
    {
        "최소 인구": (hh_region_size * lower_sizes).sum(axis=1),
        "공표 인구": person_region_age.sum(axis=1),
        "최대 인구": (hh_region_size * upper_sizes).sum(axis=1),
    }
)
feasibility.loc["전국"] = feasibility.sum()

check(
    "제약표 정합성",
    "지역별 인구와 전국 인구가 모두 가구 규모 표로 실현 가능한 범위 안에 있다",
    bool(
        (feasibility["최소 인구"] <= feasibility["공표 인구"]).all()
        and (feasibility["공표 인구"] <= feasibility["최대 인구"]).all()
    ),
)

feasibility

,최소 인구,공표 인구,최대 인구
r1,660000.0,670000.0,740000.0
r2,390000.0,398000.0,430000.0
전국,1050000.0,1068000.0,1170000.0


전국 인구 1,068,000명은 최소 1,050,000명과 최대 1,170,000명 사이에 있다. 하한과의 여유 18,000명이 개방 계급 `4+` 에 5인과 6인 가구가 들어갈 수 있는 폭이며, 11절에서 확인할 평균 가구원 수는 이 여유로부터 나온다.

## 4. 미공표 셀의 목표 구간 연역

차량 대수별 가구 수 표는 0대만 공표되었다. 남은 두 셀은 두 개의 균형식으로 좁힌다.

1. 세 셀의 합은 총 가구 수와 같다. $H_0 + H_1 + H_2 = 520{,}000$
2. 보유 차량의 총합은 등록 통계와 같다. $H_1 + 2H_2 \in [505{,}000,\ 515{,}000]$

둘째 균형식에서 0대 가구의 계수는 0이므로 그 셀은 균형식에 참여하지 않는다. `LinearBalance` 의 계수는 모두 양수여야 하므로, 참여하는 셀만 계수로 지정한다.

이 표는 미시 자료가 없을 때 결측 처리가 왜 더 중요한지를 보여 준다. 표본이 있으면 미공표 셀을 표본 분포로 메울 수 있지만, 여기서는 다른 표와의 관계 외에 기댈 근거가 없다.

In [4]:
estimator = gipu.MissingMarginEstimator(gipu.marginal_from_series(car_margin))
estimator.add_balance("households", TOTAL_HOUSEHOLDS)
estimator.add_balance("cars", CAR_TOTAL, {"1": 1.0, "2": 2.0})

car_intervals = estimator.estimate()

check(
    "주변표의 결측",
    "두 균형식으로부터 결측 셀이 유한한 폭의 구간으로 좁혀진다",
    all(np.isfinite(car_intervals[name].upper) for name in CAR_CLASSES),
)
check(
    "주변표의 결측",
    "공표된 셀은 퇴화 구간으로, 결측 셀은 폭이 있는 구간으로 남는다",
    car_intervals["0"].is_scalar and not car_intervals["2"].is_scalar,
)

estimator.to_frame()

,cell,status,lower,upper,width
0,0,observed,130000.0,130000.0,0.0
1,1,missing,265000.0,275000.0,10000.0
2,2,missing,115000.0,125000.0,10000.0


두 균형식은 $H_1 + H_2 = 390{,}000$ 과 $H_1 + 2H_2 \in [505{,}000,\ 515{,}000]$ 이므로 $H_2 \in [115{,}000,\ 125{,}000]$ 로 좁혀진다. 등록 통계의 구간 폭 10,000대가 2대 보유 가구 수의 구간 폭 10,000호로 그대로 옮겨 왔다.

셀별 구간만으로는 "세 셀의 합이 총 가구 수와 같다"는 결합 조건이 표현되지 않고, 계수가 1이 아닌 차량 총수 균형식은 이진 매핑 행렬로도 표현되지 않는다. 두 조건은 8절에서 각각 다른 방식으로 부과한다.

한편 결측 셀의 **점 추정**은 목표로 쓰지 않지만 초기 가중치를 놓는 데는 쓸 수 있다. `allocate` 는 균형식의 잔차를 결측 셀에 균등 배분한 뒤 추정 구간 안으로 잘라내므로, 배분한 값의 합이 잔차와 정확히 일치한다는 보장은 없다. 9절에서 이 값을 차량 보유의 사전 분포로만 쓰고 목표로는 쓰지 않는 이유가 이것이다.

In [5]:
car_point = estimator.allocate("households")
car_prior = pd.Series(
    {name: car_point.get(name, float(car_margin[name])) for name in CAR_CLASSES}
)

print("초기 가중치에 쓸 점 추정")
print(car_prior.map(lambda value: f"{value:,.0f}").to_string())

초기 가중치에 쓸 점 추정
0    130,000
1    265,000
2    125,000


## 5. 유형 공간의 열거

기본 단위를 만든다. 가구 하나는 **구성원 유형의 다중집합**과 **지역**, **차량 대수**로 결정된다고 본다. 구성원 유형은 연령 3계층과 성별 2계층의 조합 6가지이므로, 가구원 수 $n$ 인 가구의 구성은 6가지에서 중복을 허용해 $n$ 개를 고르는 조합이다.

여기에 도메인 규칙 두 개를 적용한다. 이 규칙들은 현실에 존재할 수 없는 조합을 걸러 내므로 **구조적 영**이며, 표본에서 시작할 때와 달리 제약 열이 아니라 **기본 단위**를 제거한다. 제거의 방식도 다르다. 제약 열은 구축한 뒤 색인을 배정하지 않지만, 기본 단위는 애초에 열거하지 않는다.

| 규칙 | 근거 |
| --- | --- |
| 미성년만으로 구성된 가구는 존재하지 않는다 | 성인 이상 구성원이 최소 한 명 있어야 한다 |
| 차량 2대는 운전 가능 구성원이 2명 이상인 가구에만 둔다 | 미성년은 차량을 운전하지 않는다 |

둘째 규칙은 자료가 아니라 가정이다. 가정을 규칙으로 넣으면 그 조합에 가중치가 배분되지 않으므로, 근거가 약한 규칙은 넣지 않는 편이 안전하다. 넣기로 했다면 그 사실을 문서에 남긴다.

In [6]:
PERSON_TYPES = [(age, sex) for age in AGE_GROUPS for sex in SEXES]
ADULT_PLUS = [index for index, (age, _) in enumerate(PERSON_TYPES) if age != "child"]

compositions = []
n_child_only = 0
for size in range(1, MAX_SIZE + 1):
    for combination in combinations_with_replacement(range(len(PERSON_TYPES)), size):
        counts = np.bincount(combination, minlength=len(PERSON_TYPES))
        if counts[ADULT_PLUS].sum() == 0:
            n_child_only += 1  # 미성년만으로 구성된 가구는 존재하지 않는다
            continue
        compositions.append(counts)
compositions = np.array(compositions)
composition_size = compositions.sum(axis=1)

enumeration = pd.DataFrame(
    {
        "가구원 수": range(1, MAX_SIZE + 1),
        "구성 조합": [
            int(comb(len(PERSON_TYPES) + size - 1, size))
            for size in range(1, MAX_SIZE + 1)
        ],
        "열거된 구성": [int((composition_size == size).sum()) for size in range(1, MAX_SIZE + 1)],
    }
)
enumeration["제외된 구성"] = enumeration["구성 조합"] - enumeration["열거된 구성"]

print(f"구성 유형 {len(compositions):,}가지 (미성년만으로 구성된 {n_child_only}가지를 제외)")
enumeration

구성 유형 896가지 (미성년만으로 구성된 27가지를 제외)


,가구원 수,구성 조합,열거된 구성,제외된 구성
0,1,6,4,2
1,2,21,18,3
2,3,56,52,4
3,4,126,121,5
4,5,252,246,6
5,6,462,455,7


구성에 지역과 차량 대수를 곱하여 기본 단위를 확정하고, 각 유형에 대응하는 구성원 행을 함께 만든다. 구성원 행은 개인 계층의 자료가 되며, 그 유형의 가구가 한 호 있을 때의 구성원 목록이다.

In [7]:
unit_records = []
member_records = []

for region in REGIONS:
    for composition_index, counts in enumerate(compositions):
        n_adult_plus = int(counts[ADULT_PLUS].sum())
        for car in CAR_CLASSES:
            if int(car) >= 2 and n_adult_plus < 2:
                continue  # 운전 가능 구성원보다 많은 차량은 두지 않는다
            hh_id = f"t{len(unit_records):05d}"
            unit_records.append(
                {
                    "hh_id": hh_id,
                    "region_id": region,
                    "car_class": car,
                    "n_cars": float(car),
                    "n_adult_plus": n_adult_plus,
                    "composition_index": composition_index,
                }
            )
            for type_index, count in enumerate(counts):
                age, sex = PERSON_TYPES[type_index]
                for _ in range(int(count)):
                    member_records.append((hh_id, region, age, sex))

households = pd.DataFrame(unit_records)
persons = pd.DataFrame(member_records, columns=["hh_id", "region_id", "age_group", "sex"])
persons.insert(0, "person_id", [f"m{index:06d}" for index in range(len(persons))])
regions = pd.DataFrame({"region_id": REGIONS})

print(f"기본 단위 {len(households):,}개, 구성원 행 {len(persons):,}개")

check(
    "구조적 영",
    "미성년만으로 구성된 유형이 열거되지 않는다",
    int((households["n_adult_plus"] == 0).sum()) == 0,
)
check(
    "구조적 영",
    "운전 가능 구성원이 2명 미만인 유형에 차량 2대가 배정되지 않는다",
    int(((households["car_class"] == "2") & (households["n_adult_plus"] < 2)).sum()) == 0,
)

households.head()

기본 단위 5,208개, 구성원 행 27,112개


,hh_id,region_id,car_class,n_cars,n_adult_plus,composition_index
0,t00000,r1,0,0.0,1,0
1,t00001,r1,1,1.0,1,0
2,t00002,r1,0,0.0,1,1
3,t00003,r1,1,1.0,1,1
4,t00004,r1,0,0.0,1,2


## 6. 계층 트리와 파생 속성

계층은 지역 - 가구 - 개인의 셋이며 기본 계층은 가구이다. 자료의 출처가 표본이 아니라 열거일 뿐, 트리를 구성하고 무결성을 검증하는 절차는 동일하다.

가구원 수 계급은 원본에 없는 값이므로 계층 구조에서 유도한다. 열거로 만든 자료에서는 이 값이 이미 알려져 있지만, 유도한 값과 열거에 쓴 값이 일치하는지 확인해 두면 열거와 트리 구성이 어긋나지 않았음을 보증할 수 있다.

In [8]:
tree = gipu.HierarchyTree(base_level="household")
tree.add_level("region", regions, primary_key="region_id")
tree.add_level(
    "household",
    households,
    primary_key="hh_id",
    parent_key="region_id",
    parent_level="region",
)
tree.add_level(
    "person",
    persons,
    primary_key="person_id",
    parent_key="hh_id",
    parent_level="household",
)
tree.validate_integrity()

member_counts = gipu.AggregationMapper.descendant_count_to_base(tree, "person")
tree.nodes["household"].data["size_class"] = gipu.AggregationMapper.size_class_labels(
    member_counts, boundaries=(1, 2, 3, 4)
).to_numpy()

check(
    "N계층 구조",
    "계층에서 유도한 가구원 수가 열거에 쓴 구성의 크기와 일치한다",
    np.array_equal(
        member_counts.to_numpy(),
        composition_size[households["composition_index"].to_numpy()].astype(float),
    ),
)

print("기본 계층과의 관계")
for level in ["region", "household", "person"]:
    print(f"  {level:10s} -> {tree.relation_to_base(level)}")
print()
print("가구원 수 계급별 유형 수")
print(tree.nodes["household"].data["size_class"].value_counts().reindex(SIZE_CLASSES).to_string())

기본 계층과의 관계
  region     -> ancestor
  household  -> base
  person     -> descendant

가구원 수 계급별 유형 수
size_class
1       16
2       92
3      288
4+    4812


## 7. 속성 행렬 구축

블록은 다섯 개이다. 열거로 만든 자료라는 사실은 이 단계에 전혀 나타나지 않는다. 속성 행렬은 기본 단위가 관측된 개체인지 열거된 유형인지 구별하지 않는다.

| 블록 | 종류 | 계층 | 제약 열 | 목표의 성격 |
| --- | --- | --- | --- | --- |
| `hh_region_size` | `CrossTabBlock` | 가구 | 지역 2 × 가구원 수 계급 4 = 8개 | 고정 스칼라 |
| `hh_car` | `CrossTabBlock` | 가구 | 차량 대수 3개 | 4절에서 연역한 구간 |
| `hh_cars_total` | `ColumnBlock` | 가구 | 총 차량 대수 1개 | 구간 |
| `person_age_sex` | `CrossTabBlock` | 개인 | 연령 3 × 성별 2 = 6개 | 고정 스칼라 |
| `person_region_age` | `CrossTabBlock` | 개인 | 지역 2 × 연령 3 = 6개 | 고정 스칼라 |

`hh_cars_total` 이 `ColumnBlock` 인 이유는 차량 총수 균형식의 계수가 1이 아니기 때문이다. 계수가 모두 1인 균형식은 `balance_hierarchy` 로 대범주 위계를 얻어 `CoarseBlock` 으로 부과할 수 있지만, 계수가 1이 아니면 이진 매핑 행렬로 표현되지 않는다. 계수를 반영한 속성 열을 만들어 그 열의 합계를 제약하는 편이 옳다. 여기서는 유형마다 보유 차량 대수를 담은 `n_cars` 열이 그 역할을 한다.

In [9]:
BLOCKS = [
    gipu.CrossTabBlock(
        name="hh_region_size",
        level="household",
        dims={"region_id": REGIONS, "size_class": SIZE_CLASSES},
    ),
    gipu.CrossTabBlock(name="hh_car", level="household", dims={"car_class": CAR_CLASSES}),
    gipu.ColumnBlock(name="hh_cars_total", level="household", cols=["n_cars"]),
    gipu.CrossTabBlock(
        name="person_age_sex",
        level="person",
        dims={"age_group": AGE_GROUPS, "sex": SEXES},
    ),
    gipu.CrossTabBlock(
        name="person_region_age",
        level="person",
        dims={"region_id": REGIONS, "age_group": AGE_GROUPS},
    ),
]

attribute = gipu.UnifiedSparseMatrixBuilder(tree).build(BLOCKS)

print(f"기본 단위 {attribute.n_units:,}개, 제약 열 {attribute.n_columns}개")
print(f"비영 성분 {attribute.nnz:,}개, "
      f"밀도 {attribute.nnz / (attribute.n_units * attribute.n_columns):.1%}")
attribute.column_frame().head(12)

기본 단위 5,208개, 제약 열 24개
비영 성분 42,166개, 밀도 33.7%


,block,column
0,hh_region_size,region_id=r1|size_class=1
1,hh_region_size,region_id=r1|size_class=2
2,hh_region_size,region_id=r1|size_class=3
3,hh_region_size,region_id=r1|size_class=4+
4,hh_region_size,region_id=r2|size_class=1
5,hh_region_size,region_id=r2|size_class=2
6,hh_region_size,region_id=r2|size_class=3
7,hh_region_size,region_id=r2|size_class=4+
8,hh_car,car_class=0
9,hh_car,car_class=1


밀도가 표본에서 시작할 때보다 훨씬 높다. 열거된 유형은 모든 범주 조합을 한 번씩 채우므로 성분이 고르게 퍼진다. 유형 수가 늘어나면 열 수는 그대로인 채 행 수만 늘어나므로, 비영 성분의 수는 유형 수에 비례하여 증가한다. 속성이 많아질수록 유형 수가 곱으로 늘어나는 점이 열거 방식의 대가이다.

## 8. 제약 등록

등록 순서는 블록의 구축 순서와 일치해야 한다. 어긋나면 목표가 엉뚱한 열에 부과되며, 열 수가 같으면 `validate_against` 도 이를 걸러 내지 못한다.

`hh_car` 의 목표는 4절에서 연역한 구간을 그대로 쓴다. 추정기의 `to_constraint` 도 같은 목표를 만들지만 제약 열의 이름이 셀 이름(`0`, `1`, `2`)을 따르므로, 속성 행렬의 열 이름(`car_class=0` 등)과 어긋난다. 목표가 올바른 열에 부과되는 데 필요한 것은 순서뿐이지만, 이름까지 맞춰 두면 보고서를 읽을 때 열을 헷갈리지 않는다. 여기서는 구간을 배열로 꺼내 열 이름과 함께 넘기고, 두 방식의 목표가 같은지 확인한다.

In [10]:
def names_of(block_name):
    return attribute.column_names[attribute.block_slices[block_name]]


registry = gipu.ConstraintRegistry()
registry.register(
    gipu.BoundChecker.from_arrays(
        "hh_region_size",
        "household",
        names_of("hh_region_size"),
        hh_region_size.to_numpy().reshape(-1),
    )
)
registry.register(
    gipu.BoundChecker.from_arrays(
        "hh_car",
        "household",
        names_of("hh_car"),
        np.array([car_intervals[name].lower for name in CAR_CLASSES]),
        np.array([car_intervals[name].upper for name in CAR_CLASSES]),
    )
)
registry.register(
    gipu.BoundChecker("hh_cars_total", "household", {"household.n_cars": CAR_TOTAL})
)
registry.register(
    gipu.BoundChecker.from_arrays(
        "person_age_sex",
        "person",
        names_of("person_age_sex"),
        person_age_sex.to_numpy().reshape(-1),
    )
)
registry.register(
    gipu.BoundChecker.from_arrays(
        "person_region_age",
        "person",
        names_of("person_region_age"),
        person_region_age.to_numpy().reshape(-1),
    )
)
registry.validate_against(attribute.n_columns)

lower, upper = registry.bounds()

check(
    "구간 목표",
    "제약 열의 이름이 등록소와 속성 행렬에서 같은 순서로 나열된다",
    [name.split("::", 1)[1] for name in registry.column_names] == attribute.column_names,
)
check(
    "주변표의 결측",
    "배열로 넘긴 차량 목표가 추정기의 to_constraint 가 만드는 목표와 같다",
    np.allclose(registry.get("hh_car").lower, estimator.to_constraint("hh_car", "household").lower)
    and np.allclose(
        registry.get("hh_car").upper, estimator.to_constraint("hh_car", "household").upper
    ),
)

pd.DataFrame(
    {
        "제약": registry.column_owners,
        "제약 열": attribute.column_names,
        "하한": lower,
        "상한": upper,
        "구간 여부": lower != upper,
    }
)

,제약,제약 열,하한,상한,구간 여부
0,hh_region_size,region_id=r1|size_class=1,120000.0,120000.0,False
1,hh_region_size,region_id=r1|size_class=2,100000.0,100000.0,False
2,hh_region_size,region_id=r1|size_class=3,60000.0,60000.0,False
3,hh_region_size,region_id=r1|size_class=4+,40000.0,40000.0,False
4,hh_region_size,region_id=r2|size_class=1,80000.0,80000.0,False
5,hh_region_size,region_id=r2|size_class=2,70000.0,70000.0,False
6,hh_region_size,region_id=r2|size_class=3,30000.0,30000.0,False
7,hh_region_size,region_id=r2|size_class=4+,20000.0,20000.0,False
8,hh_car,car_class=0,130000.0,130000.0,False
9,hh_car,car_class=1,265000.0,275000.0,True


## 9. 초기 가중치: 독립 가정

표본에서 시작할 때 초기 가중치는 추출률의 역수였다. 열거에서는 추출이라는 절차가 없으므로 다른 근거가 필요하다.

각 주변표를 독립으로 보고 확률을 곱한다. 유형 $i$ 가 지역 $g$, 가구원 수 계급 $c$, 구성 $\pi$, 차량 계급 $v$ 를 가질 때

$$w_i^{(0)} \propto P(g)\,P(c \mid g)\,\frac{1}{|c|}\,P(\pi \mid n)\,P(v)$$

이다. $P(\pi \mid n)$ 은 구성원 유형의 전국 비율을 성공 확률로 하는 다항 분포이고, $1/|c|$ 는 개방 계급 안의 가구원 수를 균등하게 나눈다는 뜻이다. 마지막에 총 가구 수에 맞추어 비례 조정하므로, 구조적 영으로 제거된 유형의 몫은 남은 유형에 자동으로 재배분된다.

이 초기값은 주변표가 함의하는 것 이상을 주장하지 않는다는 점에서 자연스러운 출발점이지만, **아무 근거 없는 선택은 아니다**. 12절에서 보듯 제약이 결정하지 못하는 결합 구조는 이 선택이 결정한다.

In [11]:
def multinomial_probability(counts, type_probability):
    # 구성원 유형의 다중집합 하나가 나올 확률. 같은 유형끼리는 구별하지 않는다.
    size = int(counts.sum())
    coefficient = factorial(size)
    for count in counts:
        coefficient //= factorial(int(count))
    return coefficient * float(np.prod(type_probability ** counts))


type_probability = person_age_sex.to_numpy().reshape(-1) / TOTAL_PERSONS
composition_probability = np.array(
    [multinomial_probability(counts, type_probability) for counts in compositions]
)
# 가구원 수를 조건으로 정규화한다. 크기별 비중은 가구 규모 표가 정한다.
for size in range(1, MAX_SIZE + 1):
    in_size = composition_size == size
    composition_probability[in_size] /= composition_probability[in_size].sum()

region_probability = hh_region_size.sum(axis=1).to_numpy() / TOTAL_HOUSEHOLDS
size_given_region = (
    hh_region_size.to_numpy() / hh_region_size.sum(axis=1).to_numpy()[:, None]
)
car_probability = car_prior.to_numpy() / car_prior.to_numpy().sum()

SIZES_IN_CLASS = np.array([1, 1, 1, MAX_SIZE - 3])  # 개방 계급 '4+' 는 4, 5, 6인을 담는다

base = tree.nodes["household"].data
region_index = base["region_id"].map({name: index for index, name in enumerate(REGIONS)}).to_numpy()
class_index = base["size_class"].map({name: index for index, name in enumerate(SIZE_CLASSES)}).to_numpy()
car_index = base["car_class"].map({name: index for index, name in enumerate(CAR_CLASSES)}).to_numpy()

independent_weights = (
    region_probability[region_index]
    * size_given_region[region_index, class_index]
    / SIZES_IN_CLASS[class_index]
    * composition_probability[base["composition_index"].to_numpy()]
    * car_probability[car_index]
)
independent_weights = independent_weights / independent_weights.sum() * TOTAL_HOUSEHOLDS

uniform_weights = np.full(attribute.n_units, TOTAL_HOUSEHOLDS / attribute.n_units)

print(f"독립 가정 초기 가중치 {independent_weights.min():.6f} ~ {independent_weights.max():,.1f}")
print(f"균등 초기 가중치       {uniform_weights[0]:,.1f}")
print(f"두 초기값의 총합       {independent_weights.sum():,.0f} / {uniform_weights.sum():,.0f}")

독립 가정 초기 가중치 0.000277 ~ 28,022.1
균등 초기 가중치       99.8
두 초기값의 총합       520,000 / 520,000


초기 가중치의 최솟값이 $10^{-4}$ 보다 작다. 엔진의 가중치 하한 `weight_floor` 는 기정값이 $10^{-5}$ 이므로, 그대로 두면 희귀한 유형이 하한에 눌려 조정되지 않는다. 열거에서는 유형 하나가 담는 가구 수가 매우 작을 수 있으므로 하한을 낮춘다.

이어서 표본 영을 확인한다. 열거된 유형 공간은 가능한 조합을 모두 담고 있으므로, 목표가 양수인 제약 열에는 반드시 기여하는 유형이 있다. 표본 영은 원리적으로 발생하지 않으며, $\varepsilon$ 평활도 필요하지 않다. 이것이 열거 방식의 분명한 이점이다.

In [12]:
zero_columns = gipu.ZeroCellResolver.detect(attribute.matrix, independent_weights, lower)

check(
    "표본 영",
    "열거된 유형 공간에는 표본 영 제약 열이 존재하지 않는다",
    len(zero_columns) == 0,
)

initial_sums = attribute.matrix.T.dot(independent_weights)
pd.DataFrame(
    {
        "제약": registry.column_owners,
        "제약 열": attribute.column_names,
        "초기 가중합": initial_sums,
        "하한": lower,
        "상한": upper,
    }
).head(12)

,제약,제약 열,초기 가중합,하한,상한
0,hh_region_size,region_id=r1|size_class=1,103877.110849,120000.0,120000.0
1,hh_region_size,region_id=r1|size_class=2,104955.370116,100000.0,100000.0
2,hh_region_size,region_id=r1|size_class=3,66831.506978,60000.0,60000.0
3,hh_region_size,region_id=r1|size_class=4+,45466.728169,40000.0,40000.0
4,hh_region_size,region_id=r2|size_class=1,69251.407232,80000.0,80000.0
5,hh_region_size,region_id=r2|size_class=2,73468.759081,70000.0,70000.0
6,hh_region_size,region_id=r2|size_class=3,33415.753489,30000.0,30000.0
7,hh_region_size,region_id=r2|size_class=4+,22733.364085,20000.0,20000.0
8,hh_car,car_class=0,148145.415472,130000.0,130000.0
9,hh_car,car_class=1,301988.731539,265000.0,275000.0


## 10. 반복 수렴

엔진과 갱신식은 표본에서 시작할 때와 같다. 허용 한계는 0.1%로 두고 최대 반복은 넉넉하게 잡는다.

열거 표본은 제약 하나에 기여하는 유형이 많고 제약들이 서로 얽혀 있어, 동시 갱신이 한 회차에 옮기는 거리가 짧다. 표본에서 시작할 때보다 회차가 많이 필요한 것은 이 성질 때문이며, 한 회차의 비용은 희소 행렬-벡터 곱 두 번이므로 회차가 늘어도 실행 시간은 크게 늘지 않는다.

In [13]:
RELATIVE_GAP = 0.001
MAX_ITERATIONS = 20000


def run(initial_weights, matrix=None, constraints=None):
    # 같은 설정으로 여러 초기값과 제약 집합을 비교하기 위해 한곳에 묶는다.
    engine = gipu.GeneralizedIPUEngine.from_registry(
        attribute.matrix if matrix is None else matrix,
        registry if constraints is None else constraints,
        relative_gap=RELATIVE_GAP,
        max_iterations=MAX_ITERATIONS,
        eta=1.0,
        weight_floor=1e-9,  # 유형 하나가 담는 가구 수는 매우 작을 수 있다
    )
    return engine.fit(initial_weights)


result = run(independent_weights)

print(result)
print()
print(f"종료 사유      : {result.termination_reason}")
print(f"소요 회차      : {result.n_iterations:,}")
print(f"최대 상대 격차 : {result.report.max_relative_gap:.6f}")
print(f"가중치 총합    : {result.weights.sum():,.0f}  (목표 {TOTAL_HOUSEHOLDS:,.0f})")

check("구간 목표", "집계표만으로 구성한 제약 집합에서 반복이 수렴한다", result.converged)
check(
    "구간 목표",
    "가중치의 총합이 총 가구 수를 허용 한계 안에서 재현한다",
    abs(result.weights.sum() - TOTAL_HOUSEHOLDS) / TOTAL_HOUSEHOLDS < RELATIVE_GAP,
)

IPUResult(termination_reason='converged', n_iterations=7504, max_relative_gap=0.000999612, n_violations=0)

종료 사유      : converged
소요 회차      : 7,504
최대 상대 격차 : 0.001000
가중치 총합    : 519,809  (목표 520,000)


True

In [14]:
history = result.tracker.to_frame()
print(f"기록 {len(history):,}건 (회차 {result.n_iterations:,} + 초기 상태)")
print(f"진단 소견: {result.tracker.diagnose()}")

history.iloc[:: max(len(history) // 10, 1)]

기록 7,505건 (회차 7,504 + 초기 상태)
진단 소견: 감소 중


,iteration,max_relative_gap,max_absolute_diff,violation_rate,n_violations,n_ratio_clipped,n_floor_clipped
0,0,0.392471,66622.391635,1.000000,24,0,0
750,750,0.027331,1551.184930,0.875000,21,0,0
1500,1500,0.015881,881.672026,0.708333,17,0,0
2250,2250,0.010240,579.268835,0.708333,17,0,0
3000,3000,0.006925,398.919834,0.333333,8,0,0
3750,3750,0.004833,283.413141,0.291667,7,0,0
4500,4500,0.003446,205.648615,0.208333,5,0,0
5250,5250,0.002495,151.422206,0.166667,4,0,0
6000,6000,0.001827,112.651988,0.083333,2,0,0
6750,6750,0.001348,84.429462,0.041667,1,0,0


## 11. 적합도와 가중치 분포

제약 열별 적합도를 먼저 본다. 판정 기준과 보고 형식은 표본에서 시작할 때와 같다.

In [15]:
report = gipu.FitReportGenerator.generate_report(
    result.weighted_sums,
    lower,
    upper,
    column_names=attribute.column_names,
    column_owners=registry.column_owners,
    relative_gap=RELATIVE_GAP,
)
summary = gipu.FitReportGenerator.summarize_by_constraint(report)

check(
    "구간 목표",
    "허용 한계를 반영한 미충족 제약 열이 하나도 없다",
    int(summary["n_unsatisfied"].sum()) == 0,
)
car_span = attribute.block_slices["hh_car"]
car_sums = result.weighted_sums[car_span]
missing_at = [CAR_CLASSES.index(name) for name in estimator.missing_cells]

check(
    "주변표의 결측",
    "결측이었던 차량 셀의 가중합이 연역된 구간 안에 놓인다",
    bool(
        np.all(car_sums[missing_at] >= lower[car_span][missing_at] - 1e-6)
        and np.all(car_sums[missing_at] <= upper[car_span][missing_at] + 1e-6)
    ),
)

summary

,constraint,n_columns,n_outside_bounds,n_unsatisfied,max_relative_gap,max_bound_violation,total_target,total_estimated
0,hh_region_size,8,8,0,0.001000,47.357076,520000.0,5.198091e+05
1,person_region_age,6,6,0,0.000150,63.520129,1068000.0,1.068095e+06
2,person_age_sex,6,6,0,0.000090,31.114818,1068000.0,1.068095e+06
3,hh_car,3,1,0,0.000002,0.210527,510000.0,5.198091e+05
4,hh_cars_total,1,0,0,0.000000,0.000000,505000.0,5.112006e+05


`hh_car` 제약의 구간 위반량이 0이 아닌 것은 공표된 0대 셀이 퇴화 구간이기 때문이다. 결측이었던 두 셀은 폭이 있는 구간을 받았으므로 그 안에 들어온 시점부터 조정이 멈춘다.

### 개방 계급의 내부 구성

공표표는 `4+` 계급의 가구 수만 알려 주고 그 안에 4인, 5인, 6인 가구가 얼마나 있는지는 알려 주지 않는다. 그런데 총 인구가 고정되어 있으므로, 다른 계급의 인원을 빼면 이 계급이 담아야 할 인원이 결정된다.

$$\bar{n}_{4+} = \frac{P - H_1 - 2H_2 - 3H_3}{H_{4+}}$$

이 값은 어떤 표에도 공표되어 있지 않으며 제약들의 관계에서 유도된다. 아래에서는 가중치로 직접 계산한 평균과 이 식으로 연역한 값을 대조한다. 초기값은 4, 5, 6인을 균등하게 나누었으므로 평균 5.0에서 출발한다.

In [16]:
def open_class_mean(weights):
    # 개방 계급에 속한 유형의 가구원 수를 가중 평균한다.
    in_open_class = (tree.nodes["household"].data["size_class"] == "4+").to_numpy()
    sizes = member_counts.to_numpy()
    return float(
        (weights[in_open_class] * sizes[in_open_class]).sum() / weights[in_open_class].sum()
    )


DEDUCED_MEAN = (
    TOTAL_PERSONS
    - float((hh_region_size[["1", "2", "3"]] * np.array([1.0, 2.0, 3.0])).to_numpy().sum())
) / float(hh_region_size["4+"].sum())

print(f"초기값의 4+ 평균 가구원 수 {open_class_mean(independent_weights):.4f}")
print(f"조정 후의 4+ 평균 가구원 수 {open_class_mean(result.weights):.4f}")
print(f"제약에서 연역한 값         {DEDUCED_MEAN:.4f}")

check(
    "개방 계급",
    "개방 계급의 평균 가구원 수가 제약에서 연역한 값으로 수렴한다",
    abs(open_class_mean(result.weights) - DEDUCED_MEAN) < 0.02,
)

초기값의 4+ 평균 가구원 수 5.0019
조정 후의 4+ 평균 가구원 수 4.3100
제약에서 연역한 값         4.3000


True

### 가중치 분포의 해석

표본에서 시작할 때 유효 표본 수는 "몇 개의 개체가 실질적으로 결과를 지탱하는가"를 뜻했다. 열거에서는 표본 자체가 없으므로 이 지표는 다른 것을 가리킨다. 여기서는 **몇 개의 유형이 모집단의 대부분을 차지하는가**이며, 값이 작다고 하여 표본이 부족하다는 뜻이 되지는 않는다.

실제로 열거된 유형의 대부분은 매우 드문 조합이므로 가중치가 작을 수밖에 없고, 유효 표본 수는 유형 수보다 훨씬 작게 나온다. `flag_concerns` 가 붙이는 주의 문구도 이 맥락에서 읽어야 한다. 열거에서 이 지표가 유용한 경우는 절대 수준이 아니라 **초기값을 바꿨을 때의 변화**를 볼 때이다.

In [17]:
distribution = gipu.WeightDistributionAnalyzer.analyze(
    result.weights, initial_weights=independent_weights
)

print(f"유형 수 {distribution.n_units:,}개 중 유효 표본 수 {distribution.effective_sample_size:.1f}")
print(f"상위 1% 유형이 차지하는 비중 {distribution.top_1_percent_share:.1%}")
print(f"가중치 범위 {distribution.minimum:.6f} ~ {distribution.maximum:,.1f}")
print()
for item in gipu.WeightDistributionAnalyzer.flag_concerns(distribution):
    print("주의:", item)

distribution.to_series()

유형 수 5,208개 중 유효 표본 수 58.1
상위 1% 유형이 차지하는 비중 64.0%
가중치 범위 0.000022 ~ 33,060.6

주의: 유효 표본 수 58.1 이 기본 단위 수 5208 의 50% 에 못 미칩니다.
주의: 상위 5% 기본 단위가 전체 가중치의 88.1% 를 차지합니다.


n_units                       5208.000000
total_weight                519809.122660
minimum                          0.000022
maximum                      33060.574260
mean                            99.809739
std                            939.308369
coefficient_of_variation         9.410989
normalized_entropy               0.613686
effective_sample_size           58.146600
top_1_percent_share              0.639802
top_5_percent_share              0.880738
n_at_floor                       0.000000
max_expansion_ratio              7.019500
min_expansion_ratio              0.037740
q01                              0.001532
q25                              0.298128
q50                              1.674466
q75                              9.441926
q99                           1849.340831
dtype: float64

## 12. 결합 구조의 비식별성

여기가 표본에서 시작하는 절차와 가장 크게 갈라지는 지점이다.

제약은 24개의 선형 등식과 부등식이고 미지수는 유형의 수만큼 있다. 해는 하나로 정해지지 않으며, 제약이 지정하지 않은 결합 구조는 초기값이 결정한다. 표본이 있으면 그 자리를 관측된 공기 빈도가 메우지만, 집계 자료만으로 시작하면 메울 자료가 없다.

이를 확인하기 위해 균등 초기값에서 같은 제약으로 다시 조정한다. 두 해는 모든 제약을 똑같이 만족하지만, 제약에 없는 표에서는 서로 다른 값을 준다.

- **지역 × 차량 대수**: 두 표의 주변합은 제약되어 있으나 결합은 제약되어 있지 않다.
- **노인 단독가구 수**: 어떤 제약 열도 이 조합을 직접 지정하지 않는다.

In [18]:
uniform_result = run(uniform_weights)

print(f"독립 가정 시작: {result.termination_reason} {result.n_iterations:,} 회차")
print(f"균등 시작    : {uniform_result.termination_reason} {uniform_result.n_iterations:,} 회차")


def region_car_table(weights):
    frame = tree.nodes["household"].data[["region_id", "car_class"]].copy()
    frame["weight"] = weights
    return frame.pivot_table(
        index="region_id", columns="car_class", values="weight", aggfunc="sum"
    )


def senior_alone(weights):
    # 노인 한 명만 사는 가구. 어떤 제약 열도 이 조합을 지정하지 않는다.
    base_frame = tree.nodes["household"].data
    first_member = persons.groupby("hh_id")["age_group"].first()
    alone = (base_frame["size_class"] == "1").to_numpy()
    is_senior = base_frame["hh_id"].map(first_member).eq("senior").to_numpy()
    return float(weights[alone & is_senior].sum())


independent_table = region_car_table(result.weights)
uniform_table = region_car_table(uniform_result.weights)
region_car_gap = float(
    ((independent_table - uniform_table).abs() / independent_table).to_numpy().max()
)
senior_gap = abs(
    senior_alone(result.weights) - senior_alone(uniform_result.weights)
) / senior_alone(result.weights)

check("결합 구조의 비식별성", "두 초기값의 해가 모두 수렴한다", uniform_result.converged)
check(
    "결합 구조의 비식별성",
    "제약되지 않은 지역×차량 결합에서 두 해가 서로 다른 값을 준다",
    region_car_gap > 0.02,
)
check(
    "결합 구조의 비식별성",
    "제약되지 않은 노인 단독가구 수에서 두 해가 서로 다른 값을 준다",
    senior_gap > 0.05,
)

print()
print(f"지역×차량 결합의 최대 상대 차이 {region_car_gap:.1%}")
print(f"노인 단독가구 수: 독립 가정 {senior_alone(result.weights):,.0f}호, "
      f"균등 {senior_alone(uniform_result.weights):,.0f}호 (차이 {senior_gap:.1%})")

pd.concat(
    {"독립 가정 시작": independent_table, "균등 시작": uniform_table}, names=["초기값"]
)

독립 가정 시작: converged 7,504 회차
균등 시작    : converged 7,873 회차

지역×차량 결합의 최대 상대 차이 4.6%
노인 단독가구 수: 독립 가정 41,780호, 균등 36,939호 (차이 11.6%)


car_class                      0              1             2
초기값      region_id                                           
독립 가정 시작 r1         78967.092480  163370.194366  77504.289418
         r2         51033.118048  105047.076332  43887.352017
균등 시작    r1         81309.354481  162328.668850  76203.352107
         r2         48690.859371  106825.223847  44446.044049

두 해 모두 제약을 만족한다. 적합도만으로는 어느 쪽이 옳은지 판정할 수 없으며, 판정할 자료도 없다. 결합 구조를 두고 다투는 상황에서 "제약을 모두 만족한다"는 사실은 근거가 되지 못한다.

그러므로 집계 자료만으로 만든 합성 결과를 쓸 때는, **결론이 어느 표에 의존하는가**를 먼저 확인해야 한다. 결론이 제약에 들어간 표에만 의존하면 초기값 선택은 문제되지 않는다. 결론이 제약에 없는 결합에 의존하면 그 결합에 대한 자료를 구해 제약으로 넣거나, 초기값 가정을 명시하고 결과의 범위를 함께 제시해야 한다.

## 13. 제약을 추가하면 무엇이 결정되는가

지역 × 차량 대수 표를 새로 구했다고 하자. 이 표를 제약으로 넣으면 그 결합은 두 초기값에서 같은 값으로 수렴한다. 반면 여전히 제약되지 않은 노인 단독가구 수는 여전히 초기값에 따라 갈린다.

곧 제약을 늘리면 결정되는 부분이 늘어날 뿐이며, 제약하지 않은 결합이 저절로 결정되지는 않는다.

In [19]:
hh_region_car = pd.DataFrame(
    [[70000.0, 160000.0, 90000.0],
     [60000.0, 110000.0, 30000.0]],
    index=REGIONS,
    columns=CAR_CLASSES,
)

# 새 표가 기존 표와 양립하는지 먼저 확인한다.
check(
    "제약표 정합성",
    "새로 추가한 지역×차량 표의 주변합이 기존 표와 양립한다",
    np.allclose(hh_region_car.sum(axis=1).to_numpy(), hh_region_size.sum(axis=1).to_numpy())
    and bool(
        np.all(hh_region_car.sum(axis=0).to_numpy() >= [car_intervals[name].lower for name in CAR_CLASSES])
        and np.all(hh_region_car.sum(axis=0).to_numpy() <= [car_intervals[name].upper for name in CAR_CLASSES])
    ),
)

extended_blocks = BLOCKS + [
    gipu.CrossTabBlock(
        name="hh_region_car",
        level="household",
        dims={"region_id": REGIONS, "car_class": CAR_CLASSES},
    )
]
extended_attribute = gipu.UnifiedSparseMatrixBuilder(tree).build(extended_blocks)

extended_registry = gipu.ConstraintRegistry()
extended_registry.extend(registry.get_all())
extended_registry.register(
    gipu.BoundChecker.from_arrays(
        "hh_region_car",
        "household",
        extended_attribute.column_names[extended_attribute.block_slices["hh_region_car"]],
        hh_region_car.to_numpy().reshape(-1),
    )
)
extended_registry.validate_against(extended_attribute.n_columns)

extended_independent = run(
    independent_weights, extended_attribute.matrix, extended_registry
)
extended_uniform = run(uniform_weights, extended_attribute.matrix, extended_registry)

extended_gap = float(
    (
        (region_car_table(extended_independent.weights) - region_car_table(extended_uniform.weights)).abs()
        / region_car_table(extended_independent.weights)
    ).to_numpy().max()
)
extended_senior_gap = abs(
    senior_alone(extended_independent.weights) - senior_alone(extended_uniform.weights)
) / senior_alone(extended_independent.weights)

check(
    "결합 구조의 비식별성",
    "제약으로 넣은 지역×차량 결합은 두 초기값에서 같은 값으로 결정된다",
    extended_gap < 0.001,
)
check(
    "결합 구조의 비식별성",
    "제약을 추가해도 제약되지 않은 노인 단독가구 수는 여전히 초기값에 따라 갈린다",
    extended_senior_gap > 0.02,
)

pd.DataFrame(
    {
        "제약 집합": ["기본 24열", "지역×차량 추가 30열"],
        "지역×차량 최대 상대 차이": [region_car_gap, extended_gap],
        "노인 단독가구 상대 차이": [senior_gap, extended_senior_gap],
    }
)

,제약 집합,지역×차량 최대 상대 차이,노인 단독가구 상대 차이
0,기본 24열,0.045897,0.115865
1,지역×차량 추가 30열,0.000016,0.079649


## 14. 합성 모집단 생성

가중치는 유형별 가구 수의 추정값이므로, 정수로 바꾸면 그대로 개체 수가 된다. 유형을 그 수만큼 복제하면 합성 가구 자료가 되고, 유형에 딸린 구성원 행을 함께 복제하면 합성 개인 자료가 된다.

`integerize` 는 소수부를 성공 확률로 삼는 확률적 반올림이므로 총합의 기댓값이 보존된다. 유형 수가 많고 가중치가 작은 열거에서는 반올림의 영향이 상대적으로 크므로, 정수화 뒤에 공표표가 재현되는지 반드시 확인한다.

In [20]:
integer_weights = gipu.DatasetExporter.integerize(result.weights, random_state=0)

print(f"실수 가중치 총합 {result.weights.sum():,.1f}")
print(f"정수 가중치 총합 {integer_weights.sum():,}")
print(f"개체를 하나 이상 가지는 유형 {int((integer_weights > 0).sum()):,} / {attribute.n_units:,}")

base_frame = tree.nodes["household"].data
unit_of_household = np.repeat(np.arange(attribute.n_units), integer_weights)

synthetic_households = base_frame.iloc[unit_of_household].reset_index(drop=True)
synthetic_households.insert(
    0, "synthetic_hh_id", [f"H{index:07d}" for index in range(len(synthetic_households))]
)

print(f"\n합성 가구 {len(synthetic_households):,}호")
synthetic_households[["synthetic_hh_id", "region_id", "size_class", "car_class"]].head()

실수 가중치 총합 519,809.1
정수 가중치 총합 519,842
개체를 하나 이상 가지는 유형 3,708 / 5,208

합성 가구 519,842호


,synthetic_hh_id,region_id,size_class,car_class
0,H0000000,r1,1,0
1,H0000001,r1,1,0
2,H0000002,r1,1,0
3,H0000003,r1,1,0
4,H0000004,r1,1,0


구성원 행은 유형별로 이어져 있으므로, 복제할 행의 색인을 누적합으로 계산하여 한 번에 뽑는다. 유형마다 반복문을 돌지 않는다.

In [21]:
# 유형별 구성원 행의 시작 위치와 개수
position_of_unit = pd.Series(np.arange(attribute.n_units), index=base_frame["hh_id"])
ordered_members = persons.assign(unit=persons["hh_id"].map(position_of_unit)).sort_values("unit")
members_per_unit = (
    ordered_members.groupby("unit").size().reindex(range(attribute.n_units), fill_value=0).to_numpy()
)
member_start = np.concatenate([[0], np.cumsum(members_per_unit)])

# 합성 가구마다 그 유형의 구성원 행을 통째로 복제한다.
repeat_counts = members_per_unit[unit_of_household]
offset = np.arange(repeat_counts.sum()) - np.repeat(
    np.concatenate([[0], np.cumsum(repeat_counts)])[:-1], repeat_counts
)
member_index = np.repeat(member_start[unit_of_household], repeat_counts) + offset

synthetic_persons = ordered_members.iloc[member_index].reset_index(drop=True)
synthetic_persons["synthetic_hh_id"] = np.repeat(
    synthetic_households["synthetic_hh_id"].to_numpy(), repeat_counts
)
synthetic_persons = synthetic_persons.drop(columns=["unit", "person_id"])
synthetic_persons.insert(
    0, "synthetic_person_id", [f"P{index:07d}" for index in range(len(synthetic_persons))]
)

print(f"합성 개인 {len(synthetic_persons):,}명")
synthetic_persons.head()

합성 개인 1,068,275명


,synthetic_person_id,hh_id,region_id,age_group,sex,synthetic_hh_id
0,P0000000,t00000,r1,adult,M,H0000000
1,P0000001,t00000,r1,adult,M,H0000001
2,P0000002,t00000,r1,adult,M,H0000002
3,P0000003,t00000,r1,adult,M,H0000003
4,P0000004,t00000,r1,adult,M,H0000004


합성 자료를 다시 집계하여 공표표를 재현하는지 확인한다. 여기서 재현되는 것은 제약으로 넣은 표뿐이며, 12절에서 본 대로 제약에 없는 결합은 초기값이 결정한 값 그대로 들어가 있다.

In [22]:
synthetic_hh_table = (
    synthetic_households.pivot_table(
        index="region_id", columns="size_class", values="synthetic_hh_id", aggfunc="count"
    )
    .reindex(index=REGIONS, columns=SIZE_CLASSES)
    .astype(float)
)
synthetic_person_table = (
    synthetic_persons.pivot_table(
        index="age_group", columns="sex", values="synthetic_person_id", aggfunc="count"
    )
    .reindex(index=AGE_GROUPS, columns=SEXES)
    .astype(float)
)

hh_error = np.abs(synthetic_hh_table.to_numpy() - hh_region_size.to_numpy()) / hh_region_size.to_numpy()
person_error = (
    np.abs(synthetic_person_table.to_numpy() - person_age_sex.to_numpy())
    / person_age_sex.to_numpy()
)

check(
    "합성 모집단",
    "합성 가구 자료가 지역×가구원 수 계급 표를 0.5% 이내로 재현한다",
    float(hh_error.max()) < 0.005,
)
check(
    "합성 모집단",
    "합성 개인 자료가 전국 연령×성별 표를 0.5% 이내로 재현한다",
    float(person_error.max()) < 0.005,
)

print(f"가구 표 최대 상대 오차 {hh_error.max():.4%}")
print(f"인구 표 최대 상대 오차 {person_error.max():.4%}")

pd.concat(
    {"합성": synthetic_hh_table, "공표": hh_region_size}, names=["구분"]
)

가구 표 최대 상대 오차 0.0783%


인구 표 최대 상대 오차 0.0782%


1         2        3       4+
구분 region_id                                      
합성 r1         119972.0   99953.0  59953.0  39990.0
   r2          79993.0   69988.0  30006.0  19987.0
공표 r1         120000.0  100000.0  60000.0  40000.0
   r2          80000.0   70000.0  30000.0  20000.0

파일로 저장할 때는 `export` 를 사용한다. 확장자로 형식을 판정하며 CSV와 Parquet을 지원한다.

```python
gipu.DatasetExporter.export(synthetic_households, "synthetic_households.csv")
gipu.DatasetExporter.export(synthetic_persons, "synthetic_persons.parquet")
```

가중치를 정수화하지 않고 유형별 가중치 그대로 내보내려면 주키를 기준으로 결합한다. 유형 표에 가중치를 붙인 형태는 개체 수준 자료보다 훨씬 작으므로, 결합 분포만 필요한 용도에는 이 편이 낫다.

In [23]:
weighted_types = gipu.DatasetExporter.attach_weights(
    base_frame, result.weights, keys=attribute.base_keys, key_column="hh_id"
)
weighted_types[["hh_id", "region_id", "size_class", "car_class", "ipu_weight"]].head()

,hh_id,region_id,size_class,car_class,ipu_weight
0,t00000,r1,1,0,15943.531168
1,t00001,r1,1,1,32102.296745
2,t00002,r1,1,0,16419.457471
3,t00003,r1,1,1,33060.574260
4,t00004,r1,1,0,3195.473516


## 15. 검증 요약

각 절에서 기록한 검증을 문제별로 모은다. `check()` 는 조건이 거짓이면 그 자리에서 예외를 발생시키므로, 이 표가 출력되었다는 것은 모든 항목이 통과했다는 뜻이다.

In [24]:
checks = pd.DataFrame(CHECKS)

print(f"검증 {len(checks)}건, 통과 {int(checks['통과'].sum())}건")
print()
print(
    checks.groupby("문제", sort=False)["통과"]
    .agg(["size", "sum"])
    .rename(columns={"size": "검증 수", "sum": "통과"})
    .to_string()
)

assert checks["통과"].all(), "통과하지 못한 검증이 있다"
checks

검증 25건, 통과 25건

             검증 수  통과
문제                   
제약표 정합성         5   5
주변표의 결측         4   4
구조적 영           2   2
N계층 구조          1   1
구간 목표           4   4
표본 영            1   1
개방 계급           1   1
결합 구조의 비식별성     5   5
합성 모집단          2   2


,문제,검증 내용,통과
0,제약표 정합성,지역별 인구의 합계가 전국 연령×성별 표의 총계와 일치한다,True
1,제약표 정합성,지역별 연령 대범주의 합계가 전국 연령별 인구와 일치한다,True
2,제약표 정합성,지역별 가구 수의 합계가 총 가구 수와 일치한다,True
3,제약표 정합성,지역별 인구와 전국 인구가 모두 가구 규모 표로 실현 가능한 범위 안에 있다,True
4,주변표의 결측,두 균형식으로부터 결측 셀이 유한한 폭의 구간으로 좁혀진다,True
5,주변표의 결측,"공표된 셀은 퇴화 구간으로, 결측 셀은 폭이 있는 구간으로 남는다",True
6,구조적 영,미성년만으로 구성된 유형이 열거되지 않는다,True
7,구조적 영,운전 가능 구성원이 2명 미만인 유형에 차량 2대가 배정되지 않는다,True
8,N계층 구조,계층에서 유도한 가구원 수가 열거에 쓴 구성의 크기와 일치한다,True
9,구간 목표,제약 열의 이름이 등록소와 속성 행렬에서 같은 순서로 나열된다,True


## 정리

집계 자료만으로 시작하는 절차는 다음과 같다.

1. 공표된 표들이 서로 양립하는지 먼저 검사한다. 총계의 일치와, 열거 공간이 그 표들을 실현할 수 있는지를 함께 본다.
2. 미공표 셀은 `MissingMarginEstimator` 로 목표 구간을 연역한다. 표본이 없으므로 다른 표와의 관계가 유일한 근거이다.
3. 가능한 속성 조합을 열거하여 기본 단위를 만든다. 도메인 규칙에 어긋나는 조합은 만들지 않는다. 이 노트북의 구조적 영은 제약 열이 아니라 기본 단위를 제거한다.
4. 계층 트리부터는 표본에서 시작할 때와 완전히 같다. 속성 행렬도, 제약 등록도, 반복도 기본 단위의 출처를 묻지 않는다.
5. 초기 가중치는 주변표의 곱으로 놓는다. 이 선택은 중립이 아니며, 제약이 결정하지 못하는 결합 구조를 결정한다.
6. 정수화하면 유형별 가중치가 곧 개체 수가 되므로, 유형을 복제하여 합성 모집단을 만든다.

이 방식의 한계는 두 가지이다.

- **유형 수가 속성 수에 대해 곱으로 늘어난다.** 속성이 늘어나면 열거가 감당하지 못하는 지점이 온다. 그때는 속성을 나누어 여러 단계로 조정하거나 열거 대신 다른 방법을 쓴다.
- **결합 구조가 자료로 뒷받침되지 않는다.** 12절과 13절에서 본 대로, 제약에 넣지 않은 결합은 초기값이 결정한다. 결론이 그런 결합에 의존한다면 결과의 신뢰 구간을 제시할 수 없으므로, 해당 결합의 자료를 구하는 편이 옳다.

범주 위계, 상위 계층 개체 수의 분수 귀속, 표본 영의 $\varepsilon$ 평활, 완화 계수의 영향은 [`01_기본_사용법.ipynb`](01_기본_사용법.ipynb)에서 다룬다. 미구현 항목은 [`docs/03_잔여_과제.md`](../docs/03_잔여_과제.md)에 정리되어 있다.